# Hands-on RAG for Education
## Notebook 1: Three ready-made ways to build a RAG assistant

ACM SIGCITE 2026 tutorial · Yong Zheng, Illinois Institute of Technology

**Lab version** for the tutorial JupyterHub.

In this notebook you try three of the four approaches from the tutorial on the same documents and the same five test questions, and you score every answer yourself:

| Approach | Tool | What you do here |
| --- | --- | --- |
| 1. Consumer platform | NotebookLM | Build your own NotebookLM notebook in the browser and ask the questions |
| 2. Managed RAG API | Gemini File Search | Use your own free Gemini API key to build a document store and ask the questions in a few lines of code |
| 3. Self-hosted application | AnythingLLM | Ask the instructor's AnythingLLM server, once without documents and once with them |

Your scores are saved to a small file. Notebook 2, where you build the fourth approach from scratch, reads that file and compares all four approaches.

**How to run:** click a cell and press **Shift + Enter**. Run the cells from top to bottom.

## 0. Setup

This cell loads the Python packages, which are already installed on the tutorial JupyterHub, and downloads the two lab documents.

In [ ]:
import os, sys, json, pathlib, urllib.request

import httpx
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from google import genai

# The two lab documents, downloaded from the tutorial website
MATERIALS_URL = "https://academictutorials.github.io/EdRAG/resources/materials/"
DOCUMENTS = {
    "ITM_Student_Handbook.pdf": "ITM Graduate Student Handbook (Fall 2025)",
    "Tuition_Fees.html": "Tuition, Fees, and Costs page (2026-27 estimates)",
}
DATA_DIR = pathlib.Path("data")
DATA_DIR.mkdir(exist_ok=True)
for name in DOCUMENTS:
    path = DATA_DIR / name
    if not path.exists():
        urllib.request.urlretrieve(MATERIALS_URL + name, path)
    print(f"{name:28s} {path.stat().st_size:>9,} bytes")

# The tutorial gateway runs on this JupyterHub server
GATEWAY = "http://127.0.0.1:8000/v1"
HUB_TOKEN = os.environ.get("JUPYTERHUB_API_TOKEN", "")
USER = os.environ.get("JUPYTERHUB_USER", "student")
if not HUB_TOKEN:
    print("\nWarning: this does not look like the tutorial JupyterHub. Use the offline (Colab) version instead.")

print("\npandas", pd.__version__, "| ready")

## 1. The five test questions and the scoring rubric

Expected answers, sources, and common mistakes for every question are on the tutorial website:
**https://academictutorials.github.io/EdRAG/edrag-test-questions.html**

Q1–Q4 have answers in the documents. **Q5 does not**: it checks whether an approach says so instead of guessing.

**Give each answer one overall correctness score:**

* **0 = incorrect**: the answer contains any wrong or invented information, even if the rest is right
* **1 = partially correct**: everything stated is correct, but some expected information is missing
* **2 = completely correct**: correct and complete

Use the five criteria below as a checklist when you decide. Later, you can also score each criterion separately with the same 0/1/2 scale (optional cells marked *Detailed scoring*).

| Criterion | What to check |
| --- | --- |
| Retrieval relevance | The retrieved passages are the ones that hold the answer (for Q5, the closest related passages). If a tool does not show what it retrieved, judge from its citations. |
| Grounding | Every statement in the answer is supported by the retrieved passages. |
| Citation accuracy | The cited document and page point to the passage the answer actually uses. |
| Absence of unsupported claims | The answer adds no facts, numbers, or advice that the documents do not contain. |
| Correct refusal | When the documents do not hold the answer, or hold only part of it, the answer says so. When they do hold it, the answer gives it. |

* Overall correctness: up to 2 points per answer, **10 points** per approach for Q1–Q5.
* Scored by criterion: up to 10 points per answer, **50 points** per approach.

In [ ]:
QUESTIONS = {
    "Q1": "What is the tuition rate for a graduate student?",
    "Q2": "What is the tuition rate for an international student?",
    "Q3": "I am having problems with Canvas. Who should I contact, and how?",
    "Q4": "Can I take two online courses this semester?",
    "Q5": "May I know Prof. Yong Zheng's office room number and office hours?",
}
CRITERIA = ["Retrieval relevance", "Grounding", "Citation accuracy", "No unsupported claims", "Correct refusal"]
APPROACH_NAMES = {
    "notebooklm": "1. NotebookLM",
    "file_search": "2. Gemini File Search",
    "anythingllm": "3. AnythingLLM",
    "from_scratch": "4. Built from scratch",
}
COLORS = {"notebooklm": "#1479b8", "file_search": "#e88a42", "anythingllm": "#2e9e6b", "from_scratch": "#822dd6"}
SCORE_FILE = pathlib.Path.home() / "my_scores.json"

for q, text in QUESTIONS.items():
    print(f"{q}: {text}")

The next cell defines the functions for your scores. You do not need to change it.

* `save_overall(approach, scores)`: one overall correctness score (0, 1, or 2) per question.
* `save_detailed(approach, scores)`: five criterion scores per question (optional).
* `compare_overall()` and `compare_detailed()`: tables and charts for every approach you have scored so far.

Scores are saved in `my_scores.json` in your home folder, so Notebook 2 can add the approach you build yourself.

In [ ]:
def _load():
    data = json.loads(SCORE_FILE.read_text()) if SCORE_FILE.exists() else {}
    return {"overall": data.get("overall", {}), "detailed": data.get("detailed", {})}


def _store(kind, approach, scores):
    data = _load()
    data[kind][approach] = scores
    SCORE_FILE.write_text(json.dumps(data, indent=2))


def save_overall(approach, scores):
    missing = [q for q in QUESTIONS if scores.get(q) not in (0, 1, 2)]
    if missing:
        print("Please give an overall score (0, 1, or 2) for:", ", ".join(missing))
        return
    _store("overall", approach, {q: scores[q] for q in QUESTIONS})
    table = pd.DataFrame({"Overall correctness": {q: scores[q] for q in QUESTIONS}})
    display(table.T)
    print(f"Saved. {APPROACH_NAMES[approach]}: {sum(scores[q] for q in QUESTIONS)} / 10 points")


def save_detailed(approach, scores):
    missing = [q for q in QUESTIONS
               if len(scores.get(q, [])) != 5 or any(v not in (0, 1, 2) for v in scores[q])]
    if missing:
        print("Please give five scores (0, 1, or 2) for:", ", ".join(missing))
        return
    _store("detailed", approach, {q: list(scores[q]) for q in QUESTIONS})
    table = pd.DataFrame({q: scores[q] for q in QUESTIONS}, index=CRITERIA).T
    table["Total"] = table.sum(axis=1)
    display(table)
    print(f"Saved. {APPROACH_NAMES[approach]}: {table['Total'].sum()} / 50 points")


def _plot(totals, by_question, total_max, question_max, by_criterion=None):
    from matplotlib.patches import Patch
    names = list(totals.index)
    colors = [COLORS[a] for a in by_question.attrs["order"]]
    n = 3 if by_criterion is not None else 2
    fig, axes = plt.subplots(1, n, figsize=(5.5 * n, 4.8))
    axes[0].bar(names, totals.values, color=colors)
    axes[0].set_title(f"Total (max {total_max})")
    axes[0].set_ylim(0, total_max * 1.12)
    axes[0].tick_params(axis="x", rotation=15)
    by_question.plot.bar(ax=axes[1], rot=0, color=colors, legend=False, title=f"By question (max {question_max})")
    axes[1].set_ylim(0, question_max * 1.18)
    axes[1].set_yticks(range(0, question_max + 1, 1 if question_max <= 2 else 2))
    if by_criterion is not None:
        by_criterion.iloc[::-1].plot.barh(ax=axes[2], color=colors, legend=False, title="By criterion (max 10)")
        axes[2].set_xlim(0, 11.5)
    # Write the score on every bar, so a score of 0 is visible as "0" instead of a missing bar
    for ax in axes:
        for container in ax.containers:
            ax.bar_label(container, fmt="%g", padding=2, fontsize=8)
    fig.legend(handles=[Patch(color=c, label=nm) for c, nm in zip(colors, names)],
               loc="lower center", ncol=len(names), frameon=False)
    plt.tight_layout(rect=(0, 0.08, 1, 1))
    plt.show()


def compare_overall():
    scores = _load()["overall"]
    order = [a for a in APPROACH_NAMES if a in scores]
    if not order:
        print("No overall scores saved yet.")
        return
    by_question = pd.DataFrame({APPROACH_NAMES[a]: scores[a] for a in order})
    by_question.attrs["order"] = order
    totals = by_question.sum()
    display(Markdown("**Overall correctness per question (max 2 each)**"))
    display(pd.concat([by_question, totals.to_frame("Total").T]))
    _plot(totals, by_question, 10, 2)


def compare_detailed():
    scores = _load()["detailed"]
    order = [a for a in APPROACH_NAMES if a in scores]
    if not order:
        print("No detailed scores saved yet.")
        return
    by_question = pd.DataFrame({APPROACH_NAMES[a]: {q: sum(scores[a][q]) for q in QUESTIONS} for a in order})
    by_question.attrs["order"] = order
    by_criterion = pd.DataFrame({APPROACH_NAMES[a]: [sum(scores[a][q][i] for q in QUESTIONS) for i in range(5)]
                                 for a in order}, index=CRITERIA)
    totals = by_question.sum()
    display(Markdown("**Points per question by criterion (max 10 each)**"))
    display(pd.concat([by_question, totals.to_frame("Total").T]))
    _plot(totals, by_question, 50, 10, by_criterion)

---
## 2. Approach 1: NotebookLM (consumer platform)

A consumer platform needs no code at all. Build your own NotebookLM notebook with the two lab documents (about 3 minutes):

1. Download the handbook PDF to your computer:
   **https://academictutorials.github.io/EdRAG/resources/materials/ITM_Student_Handbook.pdf**
2. Open **https://notebooklm.google.com** and sign in. A personal Gmail account works best; some school accounts have NotebookLM turned off.
3. Click **+ New notebook**.
4. Add the first source: **upload** the handbook PDF.
5. Add the second source: choose **Website** and paste
   **https://academictutorials.github.io/EdRAG/resources/materials/Tuition_Fees.html**
6. When both sources show as ready, ask Q1–Q5 one at a time (copy them from the list above).
7. Compare each answer with the expected answers on the website, then fill in your scores below.

**Cannot sign in or short on time?** Open the instructor's shared notebook instead (link and QR code on the slide). It has the same two sources.

Notice what you **cannot** see or change: how the documents were split, which passages were retrieved, and which model answered.

In [ ]:
# Your overall correctness score for NotebookLM, one number per question
# 0 = incorrect (any wrong or invented information), 1 = partially correct (no errors, something missing), 2 = completely correct
notebooklm_scores_overall = {"Q1": None, "Q2": None, "Q3": None, "Q4": None, "Q5": None}
save_overall("notebooklm", notebooklm_scores_overall)

*Detailed scoring (optional):* score each criterion separately for NotebookLM.

In [ ]:
# Five criterion scores per question, each 0, 1, or 2
#                  Relevance  Grounding  Citation  No-unsupported  Refusal
notebooklm_scores = {
    "Q1": [None,      None,      None,     None,           None],
    "Q2": [None,      None,      None,     None,           None],
    "Q3": [None,      None,      None,     None,           None],
    "Q4": [None,      None,      None,     None,           None],
    "Q5": [None,      None,      None,     None,           None],
}
save_detailed("notebooklm", notebooklm_scores)

---
## 3. Approach 2: Gemini File Search (managed RAG API)

With a managed RAG API, you upload documents and the provider splits them, embeds them, stores them, and retrieves from them for you. You only write a few lines of code.

### 3.1 Get your own free Gemini API key (about 2 minutes)

1. Open **https://aistudio.google.com/apikey** and sign in. A personal Gmail account works best; some school accounts block AI Studio.
2. Click **Create API key** and copy it.
3. Run the next cell and paste the key into the box. The key is not shown on screen or saved to a file.

> **No key?** Leave the box empty and press Enter. The notebook will use the instructor's document store through the tutorial gateway instead.

> On the free tier, Google may use what you send to improve its products. The two lab documents are public pages, but do not type personal information into your questions.

In [ ]:
from getpass import getpass

GEMINI_KEY = getpass("Paste your Gemini API key (or press Enter to use the instructor's store): ").strip()
USE_OWN_KEY = bool(GEMINI_KEY)
FS_MODEL = "gemini-3.5-flash-lite"   # free tier: 15 requests per minute

if USE_OWN_KEY:
    client = genai.Client(api_key=GEMINI_KEY)
    print("Using your own key with model", FS_MODEL)
else:
    print("Using the instructor's document store through the tutorial gateway.")

### 3.2 Build your document store

These few lines are the whole “build” step of a managed RAG API: create a store and upload two files. Gemini splits, embeds, and indexes them for you. This takes about 20 seconds. Running the cell again reuses the store instead of uploading twice.

In [ ]:
import time

if USE_OWN_KEY:
    store_name = f"edrag-{USER}"
    store = next((s for s in client.file_search_stores.list() if s.display_name == store_name), None)
    if store is None:
        store = client.file_search_stores.create(config={"display_name": store_name})
        for file_name in DOCUMENTS:
            t0 = time.time()
            op = client.file_search_stores.upload_to_file_search_store(
                file=str(DATA_DIR / file_name), file_search_store_name=store.name,
                config={"display_name": file_name})
            while not op.done:
                time.sleep(3)
                op = client.operations.get(op)
            print(f"Indexed {file_name} in {time.time() - t0:.0f} s")
    else:
        print("Reusing your store")
    for doc in client.file_search_stores.documents.list(parent=store.name):
        print(" ", doc.display_name, doc.state)
else:
    print("Skipped: using the instructor's store.")

### 3.3 Ask the five questions

`ask_file_search()` sends one question with the File Search tool turned on, then prints the answer and the pages it cites.

In [ ]:
def ask_file_search(question):
    if USE_OWN_KEY:
        result = client.interactions.create(
            model=FS_MODEL, input=question,
            tools=[{"type": "file_search", "file_search_store_names": [store.name]}])
        answer, cites = "", set()
        for step in result.steps:
            if step.type == "model_output":
                for block in step.content:
                    if block.type == "text":
                        answer += block.text
                        for a in block.annotations or []:
                            if a.type == "file_citation":
                                cites.add((a.file_name, getattr(a, "page_number", None)))
    else:
        r = httpx.post(f"{GATEWAY}/filesearch", timeout=120,
                       headers={"Authorization": f"Bearer {HUB_TOKEN}"}, json={"question": question})
        r.raise_for_status()
        d = r.json()
        answer = d["answer"]
        cites = {(c.get("file_name"), c.get("page_number")) for c in d["citations"]}
    display(Markdown(answer))
    print("Cited:", ", ".join(f"{f} p.{p}" if p else f for f, p in sorted(cites, key=str)) or "nothing")


for q, text in QUESTIONS.items():
    display(Markdown(f"---\n### {q}: {text}"))
    ask_file_search(text)

In [ ]:
# Your overall correctness score for Gemini File Search, one number per question
# 0 = incorrect (any wrong or invented information), 1 = partially correct (no errors, something missing), 2 = completely correct
file_search_scores_overall = {"Q1": None, "Q2": None, "Q3": None, "Q4": None, "Q5": None}
save_overall("file_search", file_search_scores_overall)

*Detailed scoring (optional):* score each criterion separately for Gemini File Search.

In [ ]:
# Five criterion scores per question, each 0, 1, or 2
#                  Relevance  Grounding  Citation  No-unsupported  Refusal
file_search_scores = {
    "Q1": [None,      None,      None,     None,           None],
    "Q2": [None,      None,      None,     None,           None],
    "Q3": [None,      None,      None,     None,           None],
    "Q4": [None,      None,      None,     None,           None],
    "Q5": [None,      None,      None,     None,           None],
}
save_detailed("file_search", file_search_scores)

**Optional.** Your store stays in your Google project until you delete it. Run this cell when you are done.

In [ ]:
if USE_OWN_KEY:
    client.file_search_stores.delete(name=store.name, config={"force": True})
    print("Deleted", store.name)

---
## 4. Approach 3: AnythingLLM (self-hosted application)

AnythingLLM runs on the tutorial server. The documents stay on that server, but the passages it retrieves are still sent to a cloud language model (here, Qwen3.8-Flash). The instructor has set up two workspaces that use the **same model**:

| Workspace | Documents | Chat mode |
| --- | --- | --- |
| `norag` (Tutorial-NoRAG) | none | Chat: the model answers from memory |
| `edrag` (Tutorial-EdRAG) | the two lab documents | Query: answers only from the documents, refuses when they have nothing |

### 4.1 Same model, with and without documents

The question below names the university. Without documents, watch whether the model admits it does not know or invents contact details.

In [ ]:
_recorded = None


def recorded_answer(workspace, question):
    # Answers recorded on the tutorial server before class, used if the live server is unavailable
    global _recorded
    if _recorded is None:
        _recorded = json.loads(urllib.request.urlopen(MATERIALS_URL + "practice_offline/anythingllm_recorded.json").read())
    for item in _recorded["with_and_without_documents"]:
        if item["workspace"] == workspace and item["question"] == question:
            return item
    q = next((k for k, v in QUESTIONS.items() if v == question), None)
    return _recorded["edrag"].get(q) if workspace == "edrag" else None


def ask_anythingllm(workspace, question):
    try:
        r = httpx.post(f"{GATEWAY}/anythingllm", timeout=120,
                       headers={"Authorization": f"Bearer {HUB_TOKEN}"},
                       json={"workspace": workspace, "question": question})
        r.raise_for_status()
        d = r.json()
        timing = f" | {d['latency_ms'] / 1000:.1f} s"
    except Exception as exc:
        d = recorded_answer(workspace, question)
        if d is None:
            print("AnythingLLM is not available right now, and there is no recorded answer for this question.", exc)
            return
        print("The AnythingLLM server is not available right now. Showing the answer recorded before class.\n")
        timing = " | recorded"
    display(Markdown(d["answer"]))
    print(f"[{d['workspace']} | {d['mode']} mode | {len(d['sources'])} passages retrieved{timing}]")
    for s in d["sources"]:
        print(f"  - {s['title']}: {s['excerpt'][:300]}")


q3_with_school = "I am a student at Illinois Institute of Technology. I am having problems with Canvas. Who should I contact, and how?"
for ws in ["norag", "edrag"]:
    display(Markdown(f"---\n### Workspace: {ws}"))
    ask_anythingllm(ws, q3_with_school)

Compare the two answers with the expected answer for Q3 on the website: **OTS Support Desk, supportdesk@illinoistech.edu, 312.567.DESK (3375)**, and ITM faculty, teaching assistants, and staff cannot help with Canvas.

### 4.2 Ask the five questions in the workspace with documents

In [ ]:
for q, text in QUESTIONS.items():
    display(Markdown(f"---\n### {q}: {text}"))
    ask_anythingllm("edrag", text)

In [ ]:
# Your overall correctness score for AnythingLLM (workspace edrag), one number per question
# 0 = incorrect (any wrong or invented information), 1 = partially correct (no errors, something missing), 2 = completely correct
anythingllm_scores_overall = {"Q1": None, "Q2": None, "Q3": None, "Q4": None, "Q5": None}
save_overall("anythingllm", anythingllm_scores_overall)

*Detailed scoring (optional):* score each criterion separately for AnythingLLM (workspace edrag).

In [ ]:
# Five criterion scores per question, each 0, 1, or 2
#                  Relevance  Grounding  Citation  No-unsupported  Refusal
anythingllm_scores = {
    "Q1": [None,      None,      None,     None,           None],
    "Q2": [None,      None,      None,     None,           None],
    "Q3": [None,      None,      None,     None,           None],
    "Q4": [None,      None,      None,     None,           None],
    "Q5": [None,      None,      None,     None,           None],
}
save_detailed("anythingllm", anythingllm_scores)

---
## 5. Compare the three approaches

Run this cell after you have saved overall scores for at least one approach. Notebook 2 adds the approach you build yourself.

In [ ]:
compare_overall()

*Detailed scoring (optional):* compare the scores by criterion.

In [ ]:
compare_detailed()

**Questions to think about**

* Which approach scored best overall, and on which question did the approaches differ most?
* Which approach let you see what it retrieved? How did that affect your scoring?
* Which approach would you trust with your own institution's documents, and why?

Your scores are saved in `my_scores.json` in your home folder on the JupyterHub.